# Sketch to Realistic Train -- ControlNet + SDXL

Upload a **pencil sketch** of your train and convert it into a photorealistic metallic model
using **ControlNet Canny** with Stable Diffusion XL.

**Workflow:**
1. Install dependencies & check GPU
2. Upload your pencil sketch
3. Load ControlNet + SDXL (with CPU offload to fit T4)
4. Generate a realistic train from the sketch
5. Optionally remove the background
6. Download results

## 1. Setup & Install Dependencies

In [ ]:
!pip install -q diffusers transformers accelerate safetensors torch Pillow opencv-python-headless rembg onnxruntime
!nvidia-smi

## 2. Imports & Helpers

In [ ]:
import torch
import gc
import os
import io
import cv2
import numpy as np
from PIL import Image
from diffusers import ControlNetModel, StableDiffusionXLControlNetPipeline
from google.colab import files
import matplotlib.pyplot as plt


def show_images(images, titles=None, cols=2, figsize=(14, 7)):
    """Display one or more PIL images side-by-side with optional titles."""
    rows = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    if not isinstance(axes, list):
        axes = axes.flatten() if hasattr(axes, "flatten") else [axes]
    for i, img in enumerate(images):
        axes[i].imshow(img)
        axes[i].axis("off")
        if titles:
            axes[i].set_title(titles[i], fontsize=11)
    for j in range(len(images), len(axes)):
        axes[j].axis("off")
    plt.tight_layout()
    plt.show()

## 3. Upload Your Pencil Sketch

Run the cell below -- a file picker dialog will open. Select your sketch image (JPG / PNG).

In [ ]:
uploaded = files.upload()  # opens a file picker dialog
sketch_filename = list(uploaded.keys())[0]
sketch_image = Image.open(io.BytesIO(uploaded[sketch_filename])).convert("RGB")
sketch_image = sketch_image.resize((1024, 1024))
sketch_image.save("/content/uploaded_sketch.png")

show_images([sketch_image], titles=["Uploaded Sketch"])

## 4. Load ControlNet + SDXL

Loads the ControlNet Canny adapter and SDXL base model. Uses **CPU offload** so model
layers are moved to GPU only during inference -- this keeps peak VRAM well under the T4's 15 GB limit.

In [ ]:
# Load ControlNet Canny adapter for SDXL
controlnet = ControlNetModel.from_pretrained(
    "diffusers/controlnet-canny-sdxl-1.0",
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
)

# Load SDXL + ControlNet pipeline
controlnet_pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    controlnet=controlnet,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
)

# CPU offload: model stays in RAM, layers move to GPU only during inference
controlnet_pipe.enable_model_cpu_offload()
controlnet_pipe.enable_attention_slicing()
controlnet_pipe.enable_vae_slicing()

print("ControlNet + SDXL ready (CPU offload mode).")

## 5. Generate Realistic Train from Sketch

Edit the `sketch_prompt` to describe the material and style you want.
Adjust `controlnet_conditioning_scale` to control how strictly the output follows your sketch
(0.0 = ignore sketch, 1.0 = follow exactly).

**Tip:** You can re-run this cell with different prompts / parameters without reloading the model.

In [ ]:
# ---- Edit prompt and parameters here ----
sketch_prompt = (
    "A realistic industrial train prototype made of brushed metal, "
    "steel and aluminum surfaces, visible rivets and welding seams, "
    "matte metallic finish, photorealistic, studio lighting, 4k"
)

sketch_negative_prompt = (
    "blurry, low quality, distorted, watermark, text, "
    "painted, colorful, cartoon, anime, sketch, drawing"
)

# How strictly the output follows the sketch (0.0 = ignore, 1.0 = strict)
controlnet_conditioning_scale = 0.8

SEED = 42  # set to None for random results
# ------------------------------------------

# Step 1: Extract Canny edges from the uploaded sketch
sketch_np = np.array(sketch_image)
canny_edges = cv2.Canny(sketch_np, 100, 200)
canny_image = Image.fromarray(canny_edges).convert("RGB")

# Step 2: Generate realistic image guided by the sketch edges
generator = (
    torch.Generator("cuda").manual_seed(SEED)
    if SEED is not None else None
)

sketch_result = controlnet_pipe(
    prompt=sketch_prompt,
    negative_prompt=sketch_negative_prompt,
    image=canny_image,
    num_inference_steps=30,
    guidance_scale=7.5,
    controlnet_conditioning_scale=controlnet_conditioning_scale,
    height=1024,
    width=1024,
    generator=generator,
).images[0]

sketch_result.save("/content/train_from_sketch.png")

show_images(
    [sketch_image, canny_image, sketch_result],
    titles=["Your Sketch", "Canny Edges", "Generated Realistic Train"],
    cols=3,
    figsize=(20, 7),
)

## 6. Remove Background (Optional)

Cleanly cut out just the train using `rembg`. Produces both a transparent-background
and a white-background version.

In [ ]:
from rembg import remove

input_image = Image.open("/content/train_from_sketch.png")

# Remove background (returns RGBA with transparent background)
no_bg = remove(input_image)
no_bg.save("/content/sketch_train_no_bg.png")

# White background version
white_bg = Image.new("RGBA", no_bg.size, (255, 255, 255, 255))
white_bg.paste(no_bg, mask=no_bg.split()[3])
final = white_bg.convert("RGB")
final.save("/content/sketch_train_white_bg.png")

show_images(
    [input_image, no_bg, final],
    titles=["Generated", "Transparent BG", "White BG"],
    cols=3,
    figsize=(20, 7),
)

## 7. Download Results

In [ ]:
output_dir = "/content"
saved_files = [
    f for f in os.listdir(output_dir)
    if f.endswith(".png") and ("sketch" in f or "train" in f)
]

print("Images available for download:")
for fname in sorted(saved_files):
    fpath = os.path.join(output_dir, fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  {fname}  ({size_kb:.0f} KB)")
    files.download(fpath)

print("\nDone! Check your browser downloads.")